# 0.Setup

In [1]:
import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9, 4)

import normet as nm

In [2]:
from normet.utils.logging import enable_default_logging, get_logger
enable_default_logging(level="INFO")
log = get_logger("tutorial.model_training")

# 1. Data: load

In [3]:
from _synth import make_my1_data
my1 = make_my1_data()
my1 = my1.set_index('date')

In [4]:
my1.columns

Index(['O3', 'NO', 'NO2', 'NOXasNO2', 'SO2', 'CO', 'PM10', 'NV10', 'V10',
       'PM2.5', 'NV2.5', 'V2.5', 'ETHANE', 'ETHENE', 'ETHYNE', 'PROPANE',
       'PROPENE', 'iBUTANE', 'nBUTANE', '1BUTENE', 't2BUTENE', 'c2BUTENE',
       'iPENTANE', 'nPENTANE', 't2PENTEN', '1PENTEN', '2MEPENT', 'ISOPRENE',
       'nHEXANE', 'nHEPTANE', 'iOCTANE', 'nOCTANE', 'BENZENE', 'TOLUENE',
       'ETHBENZ', 'mpXYLENE', 'oXYLENE', '124TMB', '135TMB', 'wd', 'ws',
       'temp', 'AT10', 'AP10', 'AT2.5', 'AP2.5', 'site', 'code', 'latitude',
       'longitude', 'location_type', 'Ox', 'NOx', 'u10', 'v10', 'd2m', 't2m',
       'blh', 'sp', 'ssrd', 'tcc', 'tp', 'rh2m', 'lat', 'lon'],
      dtype='str')

In [5]:
features_to_use = [
    "u10", "v10", "d2m", "t2m", "blh", "sp", "ssrd", "tcc", "tp", "rh2m"
]

In [6]:
# Prepare data, including adding time variables, training/testing split
df_prep=nm.prepare_data(my1, value='PM2.5', feature_names=features_to_use)

[06/20/26 22:56:35] INFO     Dropped 2411 rows with NA in target.                                    ]8;id=16073709;file:///Users/user/Documents/GitHub/normet-py/src/normet/utils/prepare.py\prepare.py]8;;\:]8;id=16073710;file:///Users/user/Documents/GitHub/normet-py/src/normet/utils/prepare.py#210\210]8;;\

                    INFO     Prepared data: 6373 rows, 17 columns                                     ]8;id=16073716;file:///Users/user/Documents/GitHub/normet-py/src/normet/utils/prepare.py\prepare.py]8;;\:]8;id=16073717;file:///Users/user/Documents/GitHub/normet-py/src/normet/utils/prepare.py#79\79]8;;\

In [7]:
?nm.prepare_data

In [8]:
df_prep.columns

Index(['u10', 'v10', 'd2m', 't2m', 'blh', 'sp', 'ssrd', 'tcc', 'tp', 'rh2m',
       'date', 'value', 'date_unix', 'day_julian', 'weekday', 'hour', 'set'],
      dtype='str')

In [9]:
# df_prep is used in-memory; no CSV written

# 2.Training model with FLAML backend (lightweight, recommended default)

## 2.1 Quick start

In [10]:
target = 'value'
predictors = [
    "u10", "v10", "d2m", "t2m", "blh", "sp", "ssrd", "tcc", "tp", "rh2m",
    "date_unix", "day_julian", "weekday", "hour"
]

In [11]:
#Use a single estimator (e.g., LightGBM) for a quick baseline.
#Below we use LightGBM with a small time budget.
flaml_quick_cfg = {
    "time_budget": 90,          # seconds for the search
    "metric": "r2",             # optimize R^2 (use "mae"/"mse" if preferred)
    "estimator_list": ["lgbm"], # single estimator keeps things fast
    "task": "regression",
    "eval_method": "auto",
    "save_model": True,
    "folder_path":'.',
    "filename": "automl"
}

model_flaml = nm.train_model(
    df=df_prep,
    value= target,
    backend="flaml",
    feature_names=predictors,
    model_config=flaml_quick_cfg,
    seed=42,
    verbose=True,
)

[06/20/26 22:56:36] INFO     Training FLAML AutoML: X shape=(4780, 14), target='value'         ]8;id=16073724;file:///Users/user/Documents/GitHub/normet-py/src/normet/backends/flaml_backend.py\flaml_backend.py]8;;\:]8;id=16073725;file:///Users/user/Documents/GitHub/normet-py/src/normet/backends/flaml_backend.py#285\285]8;;\

[06/20/26 22:57:55] INFO     FLAML best_estimator=lgbm | best_config={'n_estimators': 1149,    ]8;id=16073731;file:///Users/user/Documents/GitHub/normet-py/src/normet/backends/flaml_backend.py\flaml_backend.py]8;;\:]8;id=16073732;file:///Users/user/Documents/GitHub/normet-py/src/normet/backends/flaml_backend.py#296\296]8;;\
                             'num_leaves': 14, 'min_child_samples': 15, 'learning_rate':                           
                             np.float64(0.2061003131290804), 'log_max_bin': 10,                                    
                             'colsample_bytree': np.float64(0.9510000624084615), 'reg_alpha':                      
                             np.float64(1.162262163142946), 'reg_lambda':                                          
                             np.float64(0.0034651839263365124)}                                                    

                    INFO     Saved FLAML model to automl.joblib                                 ]8;id=16073738;file:///Users/user/Documents/GitHub/normet-py/src/normet/backends/flaml_backend.py\flaml_backend.py]8;;\:]8;id=16073739;file:///Users/user/Documents/GitHub/normet-py/src/normet/backends/flaml_backend.py#64\64]8;;\

In [12]:
model_flaml

,time_budget,-1
,task,'classification'
,n_jobs,-1
,eval_method,'auto'
,split_ratio,0.1
,n_splits,5
,auto_augment,True
,allow_label_overlap,True
,metric,'auto'
,estimator_list,'auto'
,log_file_name,''


# 2.2 Save/load FLAML model

In [13]:
# You can still save model if you didn't set save model as True in the model config during model training
nm.save_model(model_flaml,path='.',filename='automl.joblib')

                    INFO     Saved FLAML model to automl.joblib                                 ]8;id=16073744;file:///Users/user/Documents/GitHub/normet-py/src/normet/backends/flaml_backend.py\flaml_backend.py]8;;\:]8;id=16073745;file:///Users/user/Documents/GitHub/normet-py/src/normet/backends/flaml_backend.py#64\64]8;;\

'automl.joblib'

In [14]:
?nm.save_model

In [15]:
model_flaml=nm.load_model(path='.',filename='automl.joblib')

                    INFO     Loaded FLAML model from automl.joblib                             ]8;id=16073751;file:///Users/user/Documents/GitHub/normet-py/src/normet/backends/flaml_backend.py\flaml_backend.py]8;;\:]8;id=16073752;file:///Users/user/Documents/GitHub/normet-py/src/normet/backends/flaml_backend.py#138\138]8;;\

# 2.3 Model performance_FLAML

In [16]:
modStat = nm.modStats(df_prep, model_flaml)

In [17]:
modStat

,n,FAC2,MB,MGE,RMSE,NMB,NMGE,COE,IOA,r,p_level,R2,set
0,4780,0.990153,-0.000040,0.253551,0.330469,-0.000004,0.027825,0.953687,0.976843,0.999207,***,0.998415,training
1,1593,0.888050,0.153623,2.049144,2.925709,0.016698,0.222735,0.626788,0.813394,0.929593,***,0.864144,testing
2,6373,0.964639,0.038370,0.702378,1.490476,0.004201,0.076895,0.871812,0.935906,0.983112,***,0.966509,all


# 2.4 Partial Dependency_FLAML

In [18]:
pdp_value=nm.pdp(df_prep,model_flaml)

In [19]:
pdp_value.head()

,variable,value,pdp_mean,pdp_std
0,blh,32.616693,19.345874,9.574783
1,blh,71.202291,18.257101,9.641589
2,blh,109.787888,17.079692,9.567257
3,blh,148.373486,14.047617,7.889420
4,blh,186.959084,12.244408,7.785849


In [20]:
df_prep.columns

Index(['u10', 'v10', 'd2m', 't2m', 'blh', 'sp', 'ssrd', 'tcc', 'tp', 'rh2m',
       'date', 'value', 'date_unix', 'day_julian', 'weekday', 'hour', 'set'],
      dtype='str')

In [21]:
nm.pdp(df_prep,model_flaml,var_list=['blh','rh2m'])

,variable,value,pdp_mean,pdp_std
0,blh,32.616693,19.345874,9.574783
1,blh,71.202291,18.257101,9.641589
2,blh,109.787888,17.079692,9.567257
3,blh,148.373486,14.047617,7.889420
4,blh,186.959084,12.244408,7.785849
...,...,...,...,...
95,rh2m,93.271444,8.961660,7.744341
96,rh2m,94.519172,9.170470,7.782532
97,rh2m,95.766900,9.505157,7.829908
98,rh2m,97.014628,9.367317,7.753671


# 3.Training model with LightGBM backend (cross-validated hyperparameter search)

## 3.1 Quick config

In [22]:
lgb_quick_cfg = {
    "n_trials": 10,              # random hyperparameter trials; raise for a better search
    "cv_folds": 3,               # CV folds per trial
    "nrounds": 200,              # max boosting rounds
    "early_stopping_rounds": 20,
}


model_lgb = nm.train_model(
    df=df_prep,
    value=target,
    backend="lightgbm",
    feature_names=predictors,
    model_config=lgb_quick_cfg,
    seed=42,
    verbose=False,)

In [23]:
model_lgb

# 3.2 Save/load LightGBM model

In [24]:
# You can still save model if you didn't set save model as True in the model config during model training
nm.save_model(model_lgb,path='.',filename='automl_lgb.joblib')

[06/20/26 22:58:27] INFO     Saved LightGBM model to automl_lgb.joblib                           ]8;id=16073759;file:///Users/user/Documents/GitHub/normet-py/src/normet/backends/lgb_backend.py\lgb_backend.py]8;;\:]8;id=16073760;file:///Users/user/Documents/GitHub/normet-py/src/normet/backends/lgb_backend.py#141\141]8;;\

'automl_lgb.joblib'

In [25]:
model_lgb=nm.load_model(path='.',backend='lightgbm',filename='automl_lgb.joblib')

                    INFO     Loaded LightGBM model from automl_lgb.joblib                        ]8;id=16073766;file:///Users/user/Documents/GitHub/normet-py/src/normet/backends/lgb_backend.py\lgb_backend.py]8;;\:]8;id=16073767;file:///Users/user/Documents/GitHub/normet-py/src/normet/backends/lgb_backend.py#205\205]8;;\

In [26]:
model_lgb

# 3.3 Model performance_LightGBM

In [27]:
modStat = nm.modStats(df_prep, model_lgb)

In [28]:
modStat

,n,FAC2,MB,MGE,RMSE,NMB,NMGE,COE,IOA,r,p_level,R2,set
0,4780,0.913891,-0.024684,1.950670,2.803746,-0.002709,0.214069,0.643691,0.821846,0.942248,***,0.887832,training
1,1593,0.871069,0.084717,2.583736,3.633356,0.009208,0.280843,0.529422,0.764711,0.888891,***,0.790127,testing
2,6373,0.903190,0.002662,2.108912,3.032466,0.000291,0.230880,0.615112,0.807556,0.929577,***,0.864114,all


# 3.4 Partial Dependency_LightGBM

In [29]:
pdp_value=nm.pdp(df_prep,model_lgb)

In [30]:
pdp_value.head()

,variable,value,pdp_mean,pdp_std
0,blh,32.616693,16.552984,7.347846
1,blh,71.202291,16.552984,7.347846
2,blh,109.787888,15.387965,7.228210
3,blh,148.373486,12.608048,7.088992
4,blh,186.959084,11.574836,6.776579


In [31]:
pdp_value['variable'].unique()

<ArrowStringArray>
[       'blh',  'date_unix',        'u10',         'sp',        'd2m',
        'v10', 'day_julian',        't2m',       'rh2m',       'hour',
       'ssrd',    'weekday',        'tcc',         'tp']
Length: 14, dtype: str

In [32]:
nm.pdp(df_prep,model_lgb,var_list=['blh','rh2m'])

,variable,value,pdp_mean,pdp_std
0,blh,32.616693,16.552984,7.347846
1,blh,71.202291,16.552984,7.347846
2,blh,109.787888,15.387965,7.228210
3,blh,148.373486,12.608048,7.088992
4,blh,186.959084,11.574836,6.776579
...,...,...,...,...
95,rh2m,93.271444,8.667554,7.056684
96,rh2m,94.519172,8.732343,6.992795
97,rh2m,95.766900,8.799836,6.952079
98,rh2m,97.014628,8.799836,6.952079


# 3.5 GPU acceleration (optional)

LightGBM training can use GPU acceleration via `use_gpu=True`. This requires a CUDA-enabled LightGBM build (`device_type="cuda"`). On Apple Silicon, LightGBM has no GPU backend, so `use_gpu=True` logs a warning and falls back to CPU automatically — the same code is portable across machines.

In [33]:
gpu_cfg = {**lgb_quick_cfg, "n_trials": 3}

model_lgb_gpu = nm.train_model(
    df=df_prep,
    value=target,
    backend="lightgbm",
    feature_names=predictors,
    model_config=gpu_cfg,
    seed=42,
    use_gpu=True,
    verbose=False,
)

model_lgb_gpu.use_gpu

[06/20/26 22:58:34] WARNING  use_gpu=True has no effect on Apple Silicon: LightGBM does not      ]8;id=16073773;file:///Users/user/Documents/GitHub/normet-py/src/normet/backends/lgb_backend.py\lgb_backend.py]8;;\:]8;id=16073774;file:///Users/user/Documents/GitHub/normet-py/src/normet/backends/lgb_backend.py#334\334]8;;\
                             support Metal/MPS. Training will run on CPU. For GPU-accelerated                      
                             tree models on macOS, consider XGBoost with Metal support.                            

True